# Run this notebook from the repository root after placing the M5 CSV files in data/raw/.

In [3]:
from pathlib import Path
import pandas as pd

DATA_DIR = Path.cwd() / "data" / "raw"

sales = pd.read_csv(DATA_DIR / "sales_train_evaluation.csv")
calendar = pd.read_csv(DATA_DIR / "calendar.csv")

print(sales.shape)
print(sales.columns[:8].tolist(), sales.columns[-3:].tolist())
print(calendar.shape)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
(30490, 1947)
['id', 'item_id', 'dept_id', 'cat_id', 'store_id', 'state_id', 'd_1', 'd_2'] ['d_1939', 'd_1940', 'd_1941']
(1969, 14)


In [5]:
from pathlib import Path
import pandas as pd

DATA_DIR = Path.cwd() / "data" / "raw"

for name in [
    "sales_train_evaluation.csv",
    "sales_train_validation.csv",
    "calendar.csv",
    "sell_prices.csv",
    "sample_submission.csv",
]:
    path = DATA_DIR / name
    print(f"{name}: {'found' if path.exists() else 'MISSING'}")

sales = pd.read_csv(DATA_DIR / "sales_train_evaluation.csv", nrows=3)
calendar = pd.read_csv(DATA_DIR / "calendar.csv")

print("\nEvaluation file preview shape:", sales.shape)
print("Sales columns:", sales.columns[:8].tolist(), "...", sales.columns[-3:].tolist())
print("Calendar shape:", calendar.shape)
print("Calendar dates:", calendar["date"].min(), "to", calendar["date"].max())

sales_train_evaluation.csv: found
sales_train_validation.csv: found
calendar.csv: found
sell_prices.csv: found
sample_submission.csv: found

Evaluation file preview shape: (3, 1947)
Sales columns: ['id', 'item_id', 'dept_id', 'cat_id', 'store_id', 'state_id', 'd_1', 'd_2'] ... ['d_1939', 'd_1940', 'd_1941']
Calendar shape: (1969, 14)
Calendar dates: 2011-01-29 to 2016-06-19


In [6]:
import numpy as np
import pandas as pd

# Read the evaluation file, but use only the training period for grouping.
train_days = [f"d_{i}" for i in range(1, 1914)]
id_cols = ["id", "item_id", "dept_id", "cat_id", "store_id", "state_id"]

sales = pd.read_csv(DATA_DIR / "sales_train_evaluation.csv",
                    usecols=id_cols + train_days)
calendar = pd.read_csv(DATA_DIR / "calendar.csv")

# Keep complete calendar weeks so partial weeks at the start/end
# of the training period don't distort the volatility measure.
training_calendar = calendar[calendar["d"].isin(train_days)].copy()
days_by_week = (
    training_calendar.groupby("wm_yr_wk")["d"]
    .apply(list)
)
complete_weeks = [days for days in days_by_week if len(days) == 7]

weekly_demand = np.column_stack([
    sales[week_days].sum(axis=1).to_numpy()
    for week_days in complete_weeks
])

# Measure variation in positive-demand weeks; track zero-demand weeks separately.
positive = weekly_demand > 0
positive_count = positive.sum(axis=1)
positive_mean = np.divide(
    (weekly_demand * positive).sum(axis=1),
    positive_count,
    out=np.zeros(len(sales), dtype=float),
    where=positive_count > 0,
)
positive_var = np.divide(
    (((weekly_demand - positive_mean[:, None]) ** 2) * positive).sum(axis=1),
    positive_count,
    out=np.zeros(len(sales), dtype=float),
    where=positive_count > 0,
)
cv_positive_weeks = np.divide(
    np.sqrt(positive_var),
    positive_mean,
    out=np.zeros(len(sales), dtype=float),
    where=positive_mean > 0,
)

profile = sales[id_cols].copy()
profile["cv_positive_weeks"] = cv_positive_weeks
profile["zero_week_share"] = (weekly_demand == 0).mean(axis=1)

low_cut, high_cut = profile["cv_positive_weeks"].quantile([1/3, 2/3])
profile["volatility_group"] = pd.cut(
    profile["cv_positive_weeks"],
    bins=[-np.inf, low_cut, high_cut, np.inf],
    labels=["low", "moderate", "high"],
    include_lowest=True,
)

print(f"Complete weeks used: {len(complete_weeks)}")
print(f"Volatility cutoffs: low ≤ {low_cut:.3f}; "
      f"moderate ≤ {high_cut:.3f}; high > {high_cut:.3f}")
print("\nSeries per group:")
print(profile["volatility_group"].value_counts().sort_index())
print("\nMedian zero-demand week share by group:")
print(profile.groupby("volatility_group", observed=True)["zero_week_share"].median())
print("\nGroups by product category:")
print(pd.crosstab(profile["cat_id"], profile["volatility_group"]))

Complete weeks used: 273
Volatility cutoffs: low ≤ 0.520; moderate ≤ 0.611; high > 0.611

Series per group:
volatility_group
low         10164
moderate    10164
high        10162
Name: count, dtype: int64

Median zero-demand week share by group:
volatility_group
low         0.362637
moderate    0.388278
high        0.406593
Name: zero_week_share, dtype: float64

Groups by product category:
volatility_group   low  moderate  high
cat_id                                
FOODS             4519      4258  5593
HOBBIES           1634      1923  2093
HOUSEHOLD         4011      3983  2476


In [7]:
test_days = [f"d_{i}" for i in range(1914, 1942)]
actual_df = pd.read_csv(DATA_DIR / "sales_train_evaluation.csv",
                        usecols=["id"] + test_days)
actual_df = actual_df.set_index("id").loc[profile["id"]]
actual = actual_df[test_days].to_numpy(dtype=float)

# Baseline 1: repeat the most recent 7 days four times.
last_7 = sales[[f"d_{i}" for i in range(1907, 1914)]].to_numpy(dtype=float)
seasonal_naive = np.tile(last_7, (1, 4))

# Baseline 2: average the previous four matching weekdays,
# then repeat that 7-day pattern across the horizon.
last_28 = sales[[f"d_{i}" for i in range(1886, 1914)]].to_numpy(dtype=float)
four_week_mean = np.tile(last_28.reshape(len(sales), 4, 7).mean(axis=1), (1, 4))

def score_by_group(name, forecast):
    abs_error = np.abs(forecast - actual)
    rows = []
    for group in ["low", "moderate", "high"]:
        mask = profile["volatility_group"].astype(str).to_numpy() == group
        rows.append({
            "model": name,
            "volatility_group": group,
            "series": int(mask.sum()),
            "WAPE": abs_error[mask].sum() / actual[mask].sum(),
            "MAE": abs_error[mask].mean(),
        })
    return rows

results = (
    score_by_group("Seasonal naïve", seasonal_naive)
    + score_by_group("Four-week weekday mean", four_week_mean)
)
pd.DataFrame(results).round(4)

,model,volatility_group,series,WAPE,MAE
0,Seasonal naïve,low,10164,0.7406,1.6059
1,Seasonal naïve,moderate,10164,0.9989,1.0148
2,Seasonal naïve,high,10162,0.9713,1.1112
3,Four-week weekday mean,low,10164,0.6400,1.3877
4,Four-week weekday mean,moderate,10164,0.8930,0.9072
5,Four-week weekday mean,high,10162,0.8881,1.0160
